# Projeto 1 – Data Understanding
## Neonatal Vital Signs and Clinical Conditions (Grupo 5)

Este notebook segue a estrutura da fase de **Data Understanding** do CRISP-DM, a mesma usada no relatório do ano passado:

| Secção | Tarefa CRISP-DM | O que se faz aqui |
|---|---|---|
| 3.1 | Recolha inicial dos dados | Carregar os ficheiros tal como vieram |
| 3.2 | Descrição dos dados | Dimensões, colunas, tipos, episódios, período |
| 3.4 | Verificação da qualidade | Identificadores, datas, horas, decimais, códigos de erro, duplicados, nomes de patologias |
| (4.x) | Limpeza mínima | Corrigir o que impede a exploração (só para poder analisar; a preparação final é na fase seguinte) |
| 3.3 | Exploração | Análise univariada, multivariada (correlações) e bivariada (sinais × patologias, sinais × hora) |
| — | Alvos do enunciado | Tempo de internamento (regressão), patologia (classificação), perfis (clustering) |

**Nota sobre a ordem:** ao contrário do relatório do ano passado, verificamos a qualidade **antes** de explorar. Com valores como 999 ou −1 nos sinais vitais, qualquer média ou histograma feito sobre os dados brutos seria enganador.

Todas as figuras são guardadas na pasta `figuras/` para poderem ser coladas diretamente no relatório.

## 0. Configuração

Importamos as bibliotecas e definimos parâmetros globais. Os **limites fisiológicos** ficam todos neste bloco, num único sítio, porque são pressupostos da equipa (não vêm do enunciado) e têm de ser citados no relatório. Se os docentes indicarem outros valores, basta alterá-los aqui.

In [ ]:
# --- Bibliotecas ---
import re                      # expressões regulares (para normalizar textos e detetar formatos de data)
import unicodedata             # remover acentos ao comparar nomes de patologias
from pathlib import Path       # caminhos de ficheiros independentes do sistema operativo (Windows/macOS)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Aspeto dos gráficos e das tabelas ---
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# --- Pastas ---
DATA_DIR = Path(".")             # pasta onde estão os CSV (alterar se necessário, ex.: Path("dados"))
FIG_DIR = Path("figuras")        # as figuras do relatório ficam aqui
OUT_DIR = Path("dados_limpos")   # os CSV limpos ficam aqui
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

# --- Limites fisiológicos plausíveis para recém-nascidos (PRESSUPOSTO DA EQUIPA, a validar) ---
# Valores fora destes intervalos são tratados como erro do sensor / código de erro, não como medição.
LIMITES = {
    "sat_o2":    (50, 100),   # SpO2 em %: acima de 100 é impossível; abaixo de 50 é quase certamente artefacto
    "puls_rate": (50, 250),   # batimentos por minuto
    "perfusion": (0.02, 20),  # índice de perfusão em %
}
SINAIS = list(LIMITES)       # ["sat_o2", "puls_rate", "perfusion"]

def guardar(fig, nome):
    # Guarda a figura em PNG com boa resolução para o relatório
    fig.savefig(FIG_DIR / f"{nome}.png", dpi=200, bbox_inches="tight")

## 3.1 Recolha inicial dos dados

Lemos os ficheiros **todas as colunas como texto** (`dtype=str`). Isto é deliberado: se deixássemos o pandas adivinhar os tipos, valores como `"1,29"` (vírgula decimal) ou `"09/03/2024"` seriam convertidos ou rejeitados silenciosamente e nunca saberíamos que existiam. Primeiro vemos os dados tal como são, só depois os convertemos.

A função `encontrar` procura o ficheiro pelo início do nome, para funcionar quer o ficheiro se chame `1_sinais_vitais_por_hora.csv` quer `1_sinais_vitais_por_hora-1.csv`. O ficheiro 2 (sinais por dia) é opcional: se existir, é carregado; se não, é calculado mais à frente a partir do ficheiro 1.

In [ ]:
def encontrar(prefixo, obrigatorio=True):
    # Procura na DATA_DIR um CSV cujo nome começa por 'prefixo'
    ficheiros = sorted(DATA_DIR.glob(f"{prefixo}*.csv"))
    if not ficheiros:
        if obrigatorio:
            raise FileNotFoundError(f"Não encontrei {prefixo}*.csv em {DATA_DIR.resolve()}")
        return None
    return ficheiros[0]

f_hora = encontrar("1_sinais_vitais_por_hora")
f_dia  = encontrar("2_sinais_vitais_por_dia", obrigatorio=False)
f_pat  = encontrar("3_patologias_por_dia")

# Leitura "em bruto": tudo como texto, sem conversões automáticas
vit_raw = pd.read_csv(f_hora, dtype=str)
pat_raw = pd.read_csv(f_pat, dtype=str)
dia_raw = pd.read_csv(f_dia, dtype=str) if f_dia else None

print("Ficheiros carregados:")
print(f"  {f_hora.name}: {vit_raw.shape[0]} linhas x {vit_raw.shape[1]} colunas")
print(f"  {f_pat.name}: {pat_raw.shape[0]} linhas x {pat_raw.shape[1]} colunas")
print(f"  {f_dia.name if f_dia else '2_sinais_vitais_por_dia: não disponível (será calculado)'}")

## 3.2 Descrição dos dados

Primeiro olhar sobre cada tabela: as primeiras linhas e, para cada coluna, quantos valores distintos e em falta tem e alguns exemplos. A coluna de exemplos é a que costuma revelar problemas (formatos misturados, espaços, maiúsculas).

In [ ]:
def descrever(df, nome):
    # Tabela-resumo por coluna: n.º de valores distintos, em falta e exemplos
    resumo = pd.DataFrame({
        "distintos": df.nunique(),
        "em_falta": df.isna().sum(),
        "%_em_falta": (df.isna().mean() * 100).round(2),
        "exemplos": [df[c].dropna().drop_duplicates().head(6).tolist() for c in df.columns],
    })
    print(f"\n=== {nome}: {len(df)} linhas ===")
    display(df.head())
    display(resumo)

descrever(vit_raw, "Sinais vitais por hora (bruto)")
descrever(pat_raw, "Patologias por dia (bruto)")

## 3.4 Verificação da qualidade dos dados

Aqui documentamos **cada tipo de problema** com números concretos. São estes números que entram na tabela de qualidade do relatório. O ano passado o relatório limitou-se a dizer "não há nulos nem duplicados"; aqui há bastante mais para reportar.

### 3.4.1 Identificadores de episódio

O mesmo bebé aparece escrito de várias formas (`EP018`, `ep018`, `EP 018`, `" EP009"`). Se não normalizarmos, o pandas trata-os como bebés diferentes e o número de episódios fica inflacionado. A função `normalizar_episodio` tira espaços e põe tudo em maiúsculas; depois comparamos quantos episódios existem antes e depois.

In [ ]:
def normalizar_episodio(s):
    # Remove todos os espaços (início, fim e meio) e põe em maiúsculas: " ep 018" -> "EP018"
    return s.str.replace(r"\s+", "", regex=True).str.upper()

for nome, df in [("Sinais vitais", vit_raw), ("Patologias", pat_raw)]:
    bruto = df["episodio"]
    norm = normalizar_episodio(bruto)
    variantes = (pd.DataFrame({"bruto": bruto, "normalizado": norm})
                 .drop_duplicates()
                 .query("bruto != normalizado"))        # só as escritas "erradas"
    print(f"{nome}: {bruto.nunique()} códigos distintos em bruto -> {norm.nunique()} episódios reais")
    print(f"  linhas afetadas: {(bruto != norm).sum()}")
    print(f"  variantes encontradas: {sorted(variantes['bruto'].map(repr).tolist())}\n")

# Episódios que existem numa tabela e não na outra (importante para a junção mais à frente)
ep_vit = set(normalizar_episodio(vit_raw["episodio"]))
ep_pat = set(normalizar_episodio(pat_raw["episodio"]))
print("Episódios só com sinais vitais (nunca tiveram patologia registada):", sorted(ep_vit - ep_pat))
print("Episódios só com patologias (sem sinais vitais):", sorted(ep_pat - ep_vit))

### 3.4.2 Datas

As datas aparecem em dois formatos: `AAAA-MM-DD` (o formato documentado) e `DD/MM/AAAA`. Usamos expressões regulares para classificar cada valor pelo formato e contar quantos há de cada. Converter com um único `pd.to_datetime` sem indicar o formato seria arriscado: `04/03/2024` poderia ser lido como 3 de abril (formato americano).

In [ ]:
def formato_data(s):
    # Classifica cada texto de data pelo seu formato
    return np.select(
        [s.str.fullmatch(r"\d{4}-\d{2}-\d{2}"), s.str.fullmatch(r"\d{2}/\d{2}/\d{4}")],
        ["AAAA-MM-DD", "DD/MM/AAAA"],
        default="outro",
    )

def converter_data(s):
    # Converte cada formato explicitamente; o que não encaixar fica NaT (data em falta)
    s = s.str.strip()
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    pt  = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    return iso.fillna(pt)

for nome, df in [("Sinais vitais", vit_raw), ("Patologias", pat_raw)]:
    contagem = pd.Series(formato_data(df["data"].str.strip())).value_counts()
    print(f"{nome}:\n{contagem.to_string()}\n")

datas_vit = converter_data(vit_raw["data"])
print("Período coberto pelos sinais vitais:", datas_vit.min().date(), "a", datas_vit.max().date(),
      f"({(datas_vit.max() - datas_vit.min()).days + 1} dias)")

### 3.4.3 Horas

A hora deve estar entre 0 e 23 (o dicionário de dados diz que a hora 9 corresponde às 09:00–09:59). Valores como −1, 24 ou 25 não têm significado e contamo-los aqui. Mostramos também quantos registos há por hora: se os registos estiverem bem distribuídos ao longo do dia, não há horas sistematicamente em falta.

In [ ]:
hora_num = pd.to_numeric(vit_raw["hora"], errors="coerce")   # texto -> número; o que não for número fica NaN
hora_invalida = ~hora_num.between(0, 23)
print(f"Horas inválidas: {hora_invalida.sum()} registos -> valores {sorted(hora_num[hora_invalida].unique().tolist())}")

fig, ax = plt.subplots(figsize=(10, 3))
hora_num[~hora_invalida].astype(int).value_counts().sort_index().plot.bar(ax=ax, color="steelblue")
ax.set(title="Número de registos por hora do dia", xlabel="Hora", ylabel="Registos")
guardar(fig, "qualidade_registos_por_hora"); plt.show()

### 3.4.4 Sinais vitais: decimais, valores em falta e valores impossíveis

Para cada sinal contamos quatro situações distintas, porque cada uma tem uma causa e um tratamento diferente:

- **em falta**: célula vazia;
- **vírgula decimal**: `1,29` em vez de `1.29` (erro de exportação, corrige-se sem perder o valor);
- **código de erro**: 999, 99, 0 ou negativos, que são marcadores do equipamento e não medições;
- **fora do intervalo fisiológico**: qualquer outro valor fora de `LIMITES` (ex.: SpO2 de 101% ou 250%).

Os dois últimos passam a valor em falta (NaN) na limpeza, em vez de se apagar a linha toda, para não perdermos os outros dois sinais dessa hora.

In [ ]:
def para_numero(s):
    # Troca vírgula por ponto e converte para número
    return pd.to_numeric(s.str.strip().str.replace(",", ".", regex=False), errors="coerce")

CODIGOS_ERRO = [999, 99, 0, -1]   # marcadores típicos de "sem leitura" nos monitores

linhas = []
for c in SINAIS:
    bruto = vit_raw[c]
    num = para_numero(bruto)
    lo, hi = LIMITES[c]
    codigo = num.isin(CODIGOS_ERRO) | (num < 0)
    fora = ~num.between(lo, hi) & num.notna() & ~codigo
    linhas.append({
        "sinal": c,
        "em_falta": bruto.isna().sum(),
        "virgula_decimal": bruto.str.contains(",", na=False).sum(),
        "codigo_erro (999/99/0/negativo)": codigo.sum(),
        "fora_do_intervalo": fora.sum(),
        "exemplos_fora": sorted(num[fora].unique().tolist())[:8],
        "intervalo_aceite": f"[{lo}, {hi}]",
    })
qualidade_sinais = pd.DataFrame(linhas).set_index("sinal")
display(qualidade_sinais)

### 3.4.5 Duplicados

Verificamos duplicados de duas formas:

- **duplicados exatos**, linhas iguais em todas as colunas;
- **duplicados de chave**: o mesmo episódio, data e hora repetidos (ou episódio, data e patologia). Só se veem **depois** de normalizar identificadores e datas, porque `EP018` e `ep018` não parecem iguais em bruto.

In [ ]:
print("Duplicados exatos (em bruto):")
print(f"  sinais vitais: {vit_raw.duplicated().sum()}")
print(f"  patologias:    {pat_raw.duplicated().sum()}")

# Chaves normalizadas para detetar duplicados "escondidos"
chave_vit = pd.DataFrame({"ep": normalizar_episodio(vit_raw["episodio"]),
                          "data": converter_data(vit_raw["data"]),
                          "hora": hora_num})
chave_pat = pd.DataFrame({"ep": normalizar_episodio(pat_raw["episodio"]),
                          "data": converter_data(pat_raw["data"]),
                          "pat": pat_raw["patologia"].str.strip().str.lower()})
print("\nDuplicados de chave (depois de normalizar):")
print(f"  sinais vitais (episódio, data, hora): {chave_vit.duplicated().sum()}")
print(f"  patologias (episódio, data, patologia): {chave_pat.duplicated().sum()}")

### 3.4.6 Nomes das patologias

O dicionário de dados lista 13 patologias, mas no ficheiro há muitas mais escritas diferentes: sem acentos (`Doenca`), em maiúsculas (`ANEMIA`), com espaços a mais, abreviadas (`DMH`, `CMV`, `RGE`) ou incompletas (`icterícia`).

A normalização faz-se em dois passos:

1. Criamos uma chave de comparação sem acentos, em minúsculas e sem espaços extra.
2. Um dicionário (`MAPA_PATOLOGIAS`) traduz cada chave para o nome oficial. As abreviaturas estão no dicionário de forma explícita, para ficar documentado que foi uma decisão da equipa.

Qualquer nome que não esteja no mapa é assinalado para revisão manual, em vez de ser descartado sem aviso.

In [ ]:
PATOLOGIAS_OFICIAIS = [
    "Sépsis", "Hemorragia peri-intraventricular", "Lesão renal aguda", "Persistência do canal arterial",
    "Anemia", "Enterocolite necrosante", "Displasia broncopulmonar", "Retinopatia da prematuridade",
    "Icterícia neonatal", "Doença das membranas hialinas", "Doença metabólica óssea da prematuridade",
    "Citomegalovírus", "Refluxo gastroesofágico",
]

def chave_texto(s):
    # "  Doença das Membranas  " -> "doenca das membranas"
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).strip().lower()

# Cada nome oficial mapeia para si próprio...
MAPA_PATOLOGIAS = {chave_texto(p): p for p in PATOLOGIAS_OFICIAIS}
# ...e acrescentamos as abreviaturas e formas incompletas encontradas (decisão da equipa, a documentar)
MAPA_PATOLOGIAS.update({
    "dmh": "Doença das membranas hialinas",
    "cmv": "Citomegalovírus",
    "rge": "Refluxo gastroesofágico",
    "ictericia": "Icterícia neonatal",
    "sepsis": "Sépsis",
})

def normalizar_patologia(s):
    return s.map(lambda x: MAPA_PATOLOGIAS.get(chave_texto(x)) if pd.notna(x) else np.nan)

pat_norm = normalizar_patologia(pat_raw["patologia"])

# Tabela de correspondência: que escrita bruta foi convertida em quê e quantas vezes
correspondencia = (pd.DataFrame({"bruto": pat_raw["patologia"].map(repr), "normalizado": pat_norm})
                   .value_counts(dropna=False).rename("n").reset_index()
                   .sort_values(["normalizado", "n"], ascending=[True, False]))
display(correspondencia)

print(f"Escritas distintas em bruto: {pat_raw['patologia'].nunique()} -> patologias após normalizar: {pat_norm.nunique()}")
print(f"Patologia em falta (célula vazia): {pat_raw['patologia'].isna().sum()} linhas")
nao_mapeados = pat_raw.loc[pat_norm.isna() & pat_raw["patologia"].notna(), "patologia"].unique()
print("Nomes sem correspondência (rever à mão):", list(nao_mapeados) or "nenhum")
print("Patologias oficiais que nunca aparecem:", sorted(set(PATOLOGIAS_OFICIAIS) - set(pat_norm.dropna())))

### 3.4.7 Quadro-resumo da qualidade

Juntamos todos os problemas numa só tabela, pronta a passar para o relatório (secção 3.4).

In [ ]:
resumo_qualidade = pd.DataFrame([
    ("Sinais vitais", "Identificador escrito de várias formas", (vit_raw["episodio"] != normalizar_episodio(vit_raw["episodio"])).sum()),
    ("Sinais vitais", "Data em formato DD/MM/AAAA",             (formato_data(vit_raw["data"].str.strip()) == "DD/MM/AAAA").sum()),
    ("Sinais vitais", "Hora fora de 0–23",                       hora_invalida.sum()),
    ("Sinais vitais", "Vírgula decimal",                         qualidade_sinais["virgula_decimal"].sum()),
    ("Sinais vitais", "Valor em falta",                          qualidade_sinais["em_falta"].sum()),
    ("Sinais vitais", "Código de erro (999/99/0/negativo)",      qualidade_sinais["codigo_erro (999/99/0/negativo)"].sum()),
    ("Sinais vitais", "Valor fora do intervalo fisiológico",     qualidade_sinais["fora_do_intervalo"].sum()),
    ("Sinais vitais", "Duplicado (episódio, data, hora)",        chave_vit.duplicated().sum()),
    ("Patologias",    "Identificador escrito de várias formas", (pat_raw["episodio"] != normalizar_episodio(pat_raw["episodio"])).sum()),
    ("Patologias",    "Data em formato DD/MM/AAAA",             (formato_data(pat_raw["data"].str.strip()) == "DD/MM/AAAA").sum()),
    ("Patologias",    "Nome de patologia não normalizado",      (pat_raw["patologia"].notna() & (pat_raw["patologia"] != pat_norm)).sum()),
    ("Patologias",    "Patologia em falta",                     pat_raw["patologia"].isna().sum()),
    ("Patologias",    "Duplicado (episódio, data, patologia)",  chave_pat.duplicated().sum()),
], columns=["tabela", "problema", "registos"])
display(resumo_qualidade)
resumo_qualidade.to_csv(OUT_DIR / "resumo_qualidade.csv", index=False)

## Limpeza mínima para exploração

Aplicamos as correções identificadas acima. Esta limpeza serve **só para podermos explorar os dados com números corretos**; a preparação final (agregações, criação de variáveis, divisão treino/teste) pertence à fase de Data Preparation.

Decisões tomadas, a registar no relatório:

| Problema | Decisão | Porquê |
|---|---|---|
| Identificador / data mal escritos | Normalizar | O valor está certo, só a escrita varia |
| Hora inválida | Remover a linha | Não sabemos a que hora pertence |
| Vírgula decimal | Converter | O valor está certo |
| Código de erro / fora do intervalo | Passar a NaN (só nesse sinal) | Não é medição, mas os outros sinais da hora são válidos |
| Duplicado de chave nos sinais | Agregar pela mediana | Se forem iguais, fica igual; se diferirem, a mediana é robusta |
| Patologia em falta | Remover a linha | Uma linha de patologia sem patologia não tem informação |
| Duplicado de patologia | Remover | É o mesmo facto registado duas vezes |

In [ ]:
# ---------- Sinais vitais ----------
vit = pd.DataFrame({
    "episodio": normalizar_episodio(vit_raw["episodio"]),
    "data": converter_data(vit_raw["data"]),
    "hora": pd.to_numeric(vit_raw["hora"], errors="coerce"),
})
for c in SINAIS:
    v = para_numero(vit_raw[c])
    lo, hi = LIMITES[c]
    vit[c] = v.where(v.between(lo, hi))       # fora do intervalo (inclui 999, 0, negativos) -> NaN

n0 = len(vit)
vit = vit[vit["hora"].between(0, 23) & vit["data"].notna()].copy()   # remove horas/datas inválidas
vit["hora"] = vit["hora"].astype(int)
print(f"Linhas removidas por hora/data inválida: {n0 - len(vit)}")

n0 = len(vit)
vit = (vit.groupby(["episodio", "data", "hora"], as_index=False)[SINAIS].median())  # resolve duplicados
print(f"Linhas fundidas por duplicação de chave: {n0 - len(vit)}")

vit["timestamp"] = vit["data"] + pd.to_timedelta(vit["hora"], unit="h")  # instante exato do registo
vit = vit.sort_values(["episodio", "timestamp"]).reset_index(drop=True)

# ---------- Patologias ----------
pat = pd.DataFrame({
    "episodio": normalizar_episodio(pat_raw["episodio"]),
    "data": converter_data(pat_raw["data"]),
    "patologia": normalizar_patologia(pat_raw["patologia"]),
})
n0 = len(pat)
pat = pat.dropna(subset=["patologia", "data"]).drop_duplicates().reset_index(drop=True)
print(f"Linhas de patologia removidas (em falta ou duplicadas): {n0 - len(pat)}")

print(f"\nDepois da limpeza: {len(vit)} registos horários | {len(pat)} registos de patologia")
print(f"Episódios: {vit['episodio'].nunique()} com sinais vitais | {pat['episodio'].nunique()} com patologias")
print("\nValores em falta por sinal após limpeza:")
print(vit[SINAIS].isna().sum().to_string())

### Tabela diária de sinais vitais

O enunciado refere um ficheiro `2_sinais_vitais_por_dia`. Como a classificação de patologias é **por dia**, precisamos de uma linha por bebé e por dia. Se o ficheiro 2 não estiver disponível, calculamos essa tabela a partir dos registos horários.

Usamos a **mediana** (robusta a picos) e acrescentamos medidas que podem ser úteis mais tarde: mínimo, máximo, desvio-padrão e número de horas com registo. Por exemplo, um dia com quedas de saturação pode ter uma mediana normal mas um mínimo muito baixo.

In [ ]:
agg = {c: ["median", "min", "max", "std"] for c in SINAIS}
diario = vit.groupby(["episodio", "data"]).agg(agg)
diario.columns = [f"{c}_{f}" for c, f in diario.columns]     # nomes "achatados": sat_o2_median, ...
diario["horas_registadas"] = vit.groupby(["episodio", "data"]).size()
diario = diario.reset_index()
print(f"Tabela diária: {len(diario)} dias-bebé")
display(diario.head())

## 3.3 Exploração dos dados

### 3.3.1 Análise univariada: sinais vitais

Para cada sinal calculamos estatísticas descritivas (média, mediana, desvio-padrão, quartis, assimetria) e desenhamos histograma e boxplot.

Fazemos os gráficos **depois** da limpeza. Com os valores 999 ainda presentes, os eixos ficariam esticados até 999 e não se veria nada. Os outliers que restam nos boxplots são valores fisiologicamente possíveis e devem ser discutidos (são reais ou são ruído?), não removidos automaticamente.

In [ ]:
desc = vit[SINAIS].describe().T
desc["mediana"] = vit[SINAIS].median()
desc["assimetria"] = vit[SINAIS].skew()          # >0: cauda à direita; <0: cauda à esquerda
desc["%_em_falta"] = vit[SINAIS].isna().mean() * 100
display(desc.round(2))

NOMES = {"sat_o2": "Saturação O2 (%)", "puls_rate": "Frequência de pulso (bpm)", "perfusion": "Índice de perfusão (%)"}
fig, axes = plt.subplots(2, 3, figsize=(14, 6), gridspec_kw={"height_ratios": [3, 1]})
for i, c in enumerate(SINAIS):
    sns.histplot(vit[c].dropna(), bins=40, kde=True, ax=axes[0, i], color="steelblue")
    axes[0, i].set(title=NOMES[c], xlabel="")
    sns.boxplot(x=vit[c], ax=axes[1, i], color="lightsteelblue")
    axes[1, i].set(xlabel=NOMES[c])
fig.suptitle("Distribuição dos sinais vitais (registos horários, após limpeza)")
fig.tight_layout(); guardar(fig, "univariada_sinais"); plt.show()

# Contagem de outliers pelo critério IQR (o mesmo dos boxplots)
for c in SINAIS:
    q1, q3 = vit[c].quantile([.25, .75]); iqr = q3 - q1
    n = (~vit[c].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr) & vit[c].notna()).sum()
    print(f"{c}: {n} outliers pelo critério IQR ({n / vit[c].notna().sum():.1%})")

### 3.3.1 Análise univariada: episódios e cobertura temporal

Antes de modelar, importa saber **quanto dado temos por bebé**. Há bebés com 4 registos e outros com centenas. Calculamos, por episódio, o número de horas registadas, o primeiro e o último dia e a percentagem de horas preenchidas entre o primeiro e o último registo (lacunas na monitorização). O gráfico de Gantt mostra quando cada bebé esteve internado e quantos estavam em simultâneo, o que dá contexto à ocupação das 12 camas.

In [ ]:
episodios = vit.groupby("episodio").agg(
    primeiro=("timestamp", "min"),
    ultimo=("timestamp", "max"),
    horas_registadas=("timestamp", "size"),
)
episodios["horas_possiveis"] = (episodios["ultimo"] - episodios["primeiro"]).dt.total_seconds() / 3600 + 1
episodios["%_cobertura"] = (episodios["horas_registadas"] / episodios["horas_possiveis"] * 100).round(1)
display(episodios.describe().round(1))

# Gantt dos internamentos
ep_ord = episodios.sort_values("primeiro")
fig, ax = plt.subplots(figsize=(12, 9))
ax.hlines(y=range(len(ep_ord)), xmin=ep_ord["primeiro"], xmax=ep_ord["ultimo"], lw=4, color="steelblue")
ax.set_yticks(range(len(ep_ord))); ax.set_yticklabels(ep_ord.index, fontsize=7)
ax.set(title="Período de monitorização de cada episódio", xlabel="Data")
fig.tight_layout(); guardar(fig, "episodios_gantt"); plt.show()

# Bebés em simultâneo por dia (aproximação da ocupação)
ocupacao = diario.groupby("data")["episodio"].nunique()
print(f"Bebés monitorizados em simultâneo por dia: média {ocupacao.mean():.1f}, máximo {ocupacao.max()}")

### 3.3.1 Análise univariada: patologias

Cada patologia é contada de três formas, porque cada uma responde a uma pergunta diferente:

- **registos brutos**: quantas linhas no ficheiro original (com duplicados);
- **dias-bebé**: em quantos dias de internamento a patologia está presente (é a unidade da classificação diária);
- **episódios**: em quantos bebés diferentes aparece. Uma patologia que só aparece em 2 bebés não se aprende de forma fiável, por muitos dias que tenha.

Medimos também quantas patologias tem cada bebé num mesmo dia. Se for frequente haver mais do que uma, o problema de classificação é **multi-rótulo** e não multi-classe, e isso muda os modelos e as métricas a usar.

In [ ]:
freq = pat.groupby("patologia").agg(
    dias_bebe=("data", "size"),             # após a limpeza cada linha é um par (bebé, dia) único por patologia
    episodios=("episodio", "nunique"),      # n.º de bebés diferentes
)
freq.insert(0, "registos_brutos", normalizar_patologia(pat_raw["patologia"]).value_counts())  # antes da limpeza
freq = freq.sort_values("dias_bebe", ascending=False)
freq["%_dos_dias_com_patologia"] = (freq["dias_bebe"] / pat[["episodio", "data"]].drop_duplicates().shape[0] * 100).round(1)
display(freq)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
freq["dias_bebe"].sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set(title="Dias-bebé com cada patologia", xlabel="Dias-bebé")
freq["episodios"].sort_values().plot.barh(ax=axes[1], color="darkorange")
axes[1].set(title="Bebés com cada patologia", xlabel="N.º de episódios")
fig.tight_layout(); guardar(fig, "univariada_patologias"); plt.show()

# Quantas patologias por dia-bebé? -> multi-rótulo?
n_pat_dia = pat.groupby(["episodio", "data"]).size()
print("N.º de patologias no mesmo dia-bebé:")
print(n_pat_dia.value_counts().sort_index().rename("dias-bebé").to_string())
print(f"-> {(n_pat_dia > 1).mean():.1%} dos dias com patologia têm mais do que uma (problema multi-rótulo)")

### 3.3.2 Análise multivariada: correlações entre sinais

Calculamos duas matrizes de correlação:

- **Pearson** mede relações lineares e é sensível a outliers;
- **Spearman** mede relações monótonas usando ordens (rankings) e é mais robusta.

Se os dois coeficientes divergirem muito, a relação não é linear ou está a ser puxada por valores extremos. Fazemos isto ao nível horário e ao nível diário (medianas), porque a agregação pode reforçar ou esconder relações.

Com apenas 3 sinais a matriz é pequena; a utilidade principal é verificar redundância (dois sinais quase iguais não acrescentam informação ao modelo).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
paineis = [
    (vit[SINAIS], "pearson", "Pearson – horário"),
    (vit[SINAIS], "spearman", "Spearman – horário"),
    (diario[[f"{c}_median" for c in SINAIS]].set_axis(SINAIS, axis=1), "spearman", "Spearman – mediana diária"),
]
for ax, (dados, metodo, titulo) in zip(axes, paineis):
    sns.heatmap(dados.corr(method=metodo), annot=True, fmt=".2f", cmap="coolwarm",
                vmin=-1, vmax=1, square=True, ax=ax, cbar=False)
    ax.set_title(titulo)
fig.tight_layout(); guardar(fig, "multivariada_correlacoes"); plt.show()

# Gráfico de pares numa amostra (para não sobrecarregar com ~10 000 pontos)
g = sns.pairplot(vit[SINAIS].dropna().sample(min(2000, len(vit)), random_state=42),
                 corner=True, plot_kws={"s": 8, "alpha": .4})
g.figure.suptitle("Relação entre pares de sinais (amostra de 2000 registos)", y=1.02)
guardar(g.figure, "multivariada_pairplot"); plt.show()

### 3.3.3 Análise bivariada: sinais vitais por hora do dia

Os sinais vitais têm ritmo circadiano, e os cuidados (alimentação, higiene, procedimentos) acontecem a horas fixas. Se houver um padrão claro por hora, a hora deve entrar como variável nos modelos, ou então os valores devem ser comparados com a média dessa hora. A linha representa a mediana e a faixa o intervalo interquartil.

In [ ]:
por_hora = vit.groupby("hora")[SINAIS].quantile([.25, .5, .75]).unstack()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, c in zip(axes, SINAIS):
    ax.plot(por_hora.index, por_hora[(c, .5)], marker="o", color="steelblue")
    ax.fill_between(por_hora.index, por_hora[(c, .25)], por_hora[(c, .75)], alpha=.25, color="steelblue")
    ax.set(title=NOMES[c], xlabel="Hora do dia", xticks=range(0, 24, 3))
fig.suptitle("Mediana e intervalo interquartil por hora do dia")
fig.tight_layout(); guardar(fig, "bivariada_sinais_por_hora"); plt.show()

### 3.3.3 Análise bivariada: sinais vitais por patologia

Esta é a análise mais importante para a classificação: **as patologias refletem-se nos sinais vitais?** Se as distribuições forem quase iguais entre patologias, será difícil prever a patologia só com os sinais, e é melhor saber isso agora.

Juntamos a tabela diária às patologias pela chave `(episodio, data)`. Um dia com duas patologias aparece duas vezes (uma por patologia). Os dias **sem** patologia registada formam o grupo de comparação "Sem patologia registada". Mostramos só patologias com pelo menos 10 dias-bebé, para os boxplots não serem dominados por grupos de 2 ou 3 pontos.

O teste de **Kruskal-Wallis** verifica se pelo menos um grupo tem distribuição diferente dos outros. É não-paramétrico, adequado porque os sinais não são normais. Um p-valor baixo diz que há diferença, mas não diz quão grande é nem se é útil para prever. Atenção também a que dias do mesmo bebé não são independentes, o que torna os p-valores otimistas.

In [ ]:
from scipy.stats import kruskal

dia_pat = diario.merge(pat, on=["episodio", "data"], how="left")
dia_pat["patologia"] = dia_pat["patologia"].fillna("Sem patologia registada")

# Cobertura da junção: dias de patologia sem sinais vitais nesse dia
sem_sinais = pat.merge(diario[["episodio", "data"]], on=["episodio", "data"], how="left", indicator=True)
print(f"Registos de patologia sem sinais vitais no mesmo dia: {(sem_sinais['_merge'] == 'left_only').sum()} de {len(pat)}")
print(f"Dias-bebé com sinais vitais e sem patologia registada: "
      f"{(dia_pat.drop_duplicates(['episodio','data'])['patologia'] == 'Sem patologia registada').sum()} de {len(diario)}\n")

contagem = dia_pat["patologia"].value_counts()
grupos_ok = contagem[contagem >= 10].index
dp = dia_pat[dia_pat["patologia"].isin(grupos_ok)]
ordem = dp.groupby("patologia")["sat_o2_median"].median().sort_values().index

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
for ax, c in zip(axes, SINAIS):
    col = f"{c}_median"
    sns.boxplot(data=dp, y="patologia", x=col, order=ordem, ax=ax, color="lightsteelblue", fliersize=2)
    grupos = [g[col].dropna() for _, g in dp.groupby("patologia")]
    p = kruskal(*grupos).pvalue
    ax.set(title=f"{NOMES[c]}\nKruskal-Wallis p = {p:.2g}", xlabel="Mediana diária", ylabel="")
fig.tight_layout(); guardar(fig, "bivariada_sinais_por_patologia"); plt.show()

display(dp.groupby("patologia")[[f"{c}_median" for c in SINAIS]].median().round(2).loc[ordem])

### 3.3.3 Análise bivariada: evolução temporal de alguns bebés

Gráficos individuais ajudam a perceber o que os números escondem: quedas pontuais de saturação, lacunas na monitorização ou mudanças de patamar quando surge uma patologia. Escolhemos os 4 bebés com mais horas registadas e marcamos a sombreado os dias com patologia registada.

In [ ]:
top4 = episodios["horas_registadas"].nlargest(4).index
fig, axes = plt.subplots(len(top4), 1, figsize=(14, 3 * len(top4)), sharex=False)
for ax, ep in zip(axes, top4):
    d = vit[vit["episodio"] == ep]
    ax.plot(d["timestamp"], d["sat_o2"], lw=.8, color="steelblue", label="SpO2 (%)")
    ax2 = ax.twinx()                                            # segundo eixo para a frequência de pulso
    ax2.plot(d["timestamp"], d["puls_rate"], lw=.8, color="darkorange", alpha=.7, label="Pulso (bpm)")
    for dia in pat.loc[pat["episodio"] == ep, "data"].unique():  # sombrear dias com patologia
        ax.axvspan(dia, dia + pd.Timedelta(days=1), color="red", alpha=.06)
    pats = ", ".join(sorted(pat.loc[pat["episodio"] == ep, "patologia"].unique())) or "sem patologia registada"
    ax.set_title(f"{ep} – {pats}", fontsize=9)
    ax.set_ylabel("SpO2 (%)"); ax2.set_ylabel("Pulso (bpm)")
fig.tight_layout(); guardar(fig, "bivariada_series_temporais"); plt.show()

## Alvos do enunciado

O enunciado define três objetivos de data mining. Explorar já os alvos permite fixar métricas e *baselines* realistas antes de modelar.

### Regressão: tempo de internamento

Não existe uma coluna "tempo de internamento", por isso calculamo-lo como **dias entre o primeiro e o último registo de sinais vitais + 1**. É um pressuposto que já está no vosso Business Understanding: os registos podem começar depois da admissão ou acabar antes da alta.

Calculamos também o **baseline** que o critério CS2 usa: prever sempre a mediana do internamento e medir o MAE. Qualquer modelo terá de bater este valor.

> ⚠️ **Fuga de informação (leakage).** Qualquer variável calculada sobre **todo** o internamento (n.º total de horas registadas, n.º de dias com patologia, médias de todo o episódio) contém o próprio alvo. O gráfico abaixo mostra a correlação entre as horas registadas e o internamento precisamente para tornar isto visível. Na Data Preparation, as variáveis para a regressão devem vir só das primeiras 24 ou 48 h. Foi um erro deste tipo (usar productivity para prever stress) que inflacionou os resultados do relatório do ano passado.

In [ ]:
los = pd.DataFrame({
    "internamento_dias": (episodios["ultimo"].dt.normalize() - episodios["primeiro"].dt.normalize()).dt.days + 1,
    "horas_registadas": episodios["horas_registadas"],
    "n_patologias_distintas": pat.groupby("episodio")["patologia"].nunique(),
}).fillna({"n_patologias_distintas": 0}).astype({"n_patologias_distintas": int})

display(los["internamento_dias"].describe().round(2).to_frame().T)

mediana = los["internamento_dias"].median()
mae_baseline = (los["internamento_dias"] - mediana).abs().mean()
print(f"Baseline (prever sempre a mediana = {mediana:.0f} dias): MAE = {mae_baseline:.2f} dias")
print(f"Meta CS2 (−25%): MAE ≤ {mae_baseline * .75:.2f} dias")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(los["internamento_dias"], bins=15, ax=axes[0], color="steelblue")
axes[0].axvline(mediana, color="red", ls="--", label=f"mediana = {mediana:.0f}"); axes[0].legend()
axes[0].set(title="Distribuição do tempo de internamento", xlabel="Dias")
sns.scatterplot(data=los, x="horas_registadas", y="internamento_dias", ax=axes[1])
axes[1].set(title=f"Leakage: horas registadas vs internamento\n(Spearman = "
                  f"{los[['horas_registadas','internamento_dias']].corr('spearman').iloc[0,1]:.2f})")
sns.boxplot(data=los, x="n_patologias_distintas", y="internamento_dias", ax=axes[2], color="lightsteelblue")
axes[2].set(title="Internamento por n.º de patologias distintas\n(também é leakage: conta o internamento todo)", xlabel="N.º de patologias")
fig.tight_layout(); guardar(fig, "alvo_regressao_internamento"); plt.show()

### Clustering: perfis de patologias por bebé

Para o clustering "em termos de condições clínicas", a unidade natural é o bebé. Construímos uma matriz bebé × patologia com a **percentagem dos dias de internamento em que cada patologia esteve presente**.

O heatmap já dá uma primeira ideia de grupos: bebés só com doença das membranas hialinas, bebés com várias patologias crónicas, bebés sem patologias registadas. Também mostra que a matriz é esparsa, o que vai pesar na escolha da distância (por exemplo, Jaccard em vez de euclidiana). O dendrograma no topo agrupa bebés com perfis semelhantes e é só exploratório; o clustering propriamente dito fica para a fase de Modeling.

In [ ]:
dias_por_ep = diario.groupby("episodio").size()
perfil = (pat.groupby(["episodio", "patologia"]).size().unstack(fill_value=0)
          .reindex(dias_por_ep.index, fill_value=0)          # inclui bebés sem patologia (linha de zeros)
          .div(dias_por_ep, axis=0).clip(upper=1) * 100)      # % de dias com cada patologia
perfil = perfil[perfil.sum().sort_values(ascending=False).index]  # colunas da mais à menos frequente

print(f"Matriz de perfis: {perfil.shape[0]} bebés x {perfil.shape[1]} patologias")
print(f"Esparsidade: {(perfil == 0).mean().mean():.0%} das células são zero")
print(f"Bebés sem nenhuma patologia registada: {(perfil.sum(axis=1) == 0).sum()}")

g = sns.clustermap(perfil, cmap="Reds", col_cluster=False, figsize=(12, 12),
                   cbar_kws={"label": "% de dias com a patologia"}, yticklabels=True,
                   metric="euclidean", method="ward")
g.ax_heatmap.set(xlabel="", ylabel="Episódio")
g.ax_heatmap.tick_params(axis="y", labelsize=7)
g.figure.suptitle("Perfil clínico de cada bebé (agrupamento hierárquico exploratório)", y=1.01)
guardar(g.figure, "alvo_clustering_perfis"); plt.show()

# Co-ocorrência: que patologias aparecem nos mesmos bebés?
presenca = (perfil > 0).astype(int)
co = presenca.T @ presenca                      # n.º de bebés que têm as duas patologias
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(co, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
ax.set_title("Co-ocorrência de patologias (n.º de bebés com ambas)")
fig.tight_layout(); guardar(fig, "alvo_coocorrencia_patologias"); plt.show()

## Exportação

Guardamos as tabelas limpas para a fase de Data Preparation. **Os ficheiros originais não são alterados**, como exige o vosso BU. Exportamos:

- os registos horários limpos;
- a tabela diária;
- as patologias normalizadas;
- a tabela de episódios com o tempo de internamento;
- a matriz de perfis.

In [ ]:
exportar = {
    "sinais_vitais_hora_limpo.csv": vit,
    "sinais_vitais_dia.csv": diario,
    "patologias_limpo.csv": pat,
    "episodios_internamento.csv": episodios.join(los.drop(columns="horas_registadas")).reset_index(),
    "perfil_patologias_por_bebe.csv": perfil.round(2).reset_index(),
}
for nome, df in exportar.items():
    df.to_csv(OUT_DIR / nome, index=False)
    print(f"{OUT_DIR / nome}: {df.shape}")
print(f"\nFiguras guardadas em: {FIG_DIR.resolve()}")

## Checklist para o relatório (secção 3)

- [ ] **3.1**: ficheiros, n.º de linhas e colunas, período coberto, n.º de episódios.
- [ ] **3.2**: dicionário de dados (está no ficheiro Diretrizes) e tipos de dados após conversão.
- [ ] **3.3**: figuras `univariada_*`, `multivariada_*`, `bivariada_*` e `alvo_*`, cada uma com 2–3 observações. **Confirmem que o texto diz o mesmo que a figura**; foi aí que o relatório do ano passado mais falhou.
- [ ] **3.4**: tabela `resumo_qualidade.csv`, limites fisiológicos assumidos e decisões de limpeza.
- [ ] Decisões que passam para a Data Preparation:
  - janela inicial para a regressão (anti-leakage);
  - classificação multi-rótulo vs multi-classe;
  - agrupar patologias raras;
  - métrica de distância no clustering.